In [1]:
```python
# ============================================================
# LORD OF THE RINGS (LOTR.csv)
# NLP + TF-IDF + K-MEANS CLUSTERING
# ============================================================

# ============================================================
# 1. INSTALL / IMPORT LIBRARIES
# ============================================================

# If required, uncomment and run:
# !pip install pandas numpy matplotlib scikit-learn nltk

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import warnings

warnings.filterwarnings("ignore")

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Machine Learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

print("Libraries imported successfully.")


# ============================================================
# 2. DOWNLOAD NLTK RESOURCES
# ============================================================

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

print("NLTK resources downloaded.")


# ============================================================
# 3. LOAD THE LOTR CSV DATASET
# ============================================================

# Make sure LOTR.csv is in the same folder as your notebook.
df = pd.read_csv("LOTR.csv")

print("Dataset loaded successfully.")
print()
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())


# ============================================================
# 4. BASIC DATASET INFORMATION
# ============================================================

print("\nDataset information:")
print(df.info())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())


# ============================================================
# 5. IDENTIFY THE TEXT COLUMN
# ============================================================

# Common names used for dialogue/text columns.
possible_text_columns = [
    "dialogue",
    "Dialogue",
    "text",
    "Text",
    "line",
    "Line",
    "quote",
    "Quote",
    "speech",
    "Speech"
]

text_column = None

for column in possible_text_columns:
    if column in df.columns:
        text_column = column
        break

# If no standard text column is found, look for object/string columns.
if text_column is None:
    text_columns = df.select_dtypes(include=["object"]).columns.tolist()

    if len(text_columns) > 0:
        print("\nPossible text columns found:")
        print(text_columns)

        # Use the longest average string column as a reasonable default.
        average_lengths = {}

        for column in text_columns:
            average_lengths[column] = (
                df[column]
                .fillna("")
                .astype(str)
                .str.len()
                .mean()
            )

        text_column = max(
            average_lengths,
            key=average_lengths.get
        )

if text_column is None:
    raise ValueError(
        "No text column was found. Please set text_column manually."
    )

print("\nText column selected:", text_column)


# ============================================================
# 6. HANDLE MISSING TEXT
# ============================================================

df[text_column] = df[text_column].fillna("")

# Convert text to string
df[text_column] = df[text_column].astype(str)

# Remove rows where text is empty
df = df[df[text_column].str.strip() != ""].copy()

print("\nDataset shape after removing empty text:")
print(df.shape)


# ============================================================
# 7. INITIAL TEXT EXAMPLES
# ============================================================

print("\nExample dialogue:")
for text in df[text_column].head(5):
    print("-", text)


# ============================================================
# 8. NLP PREPROCESSING
# ============================================================

# English stopwords
stop_words = set(stopwords.words("english"))

# Add some common LOTR-specific words if desired.
# These words may occur frequently without helping clustering.
custom_stopwords = {
    "said",
    "say",
    "says",
    "shall",
    "would",
    "could",
    "one",
    "like"
}

stop_words.update(custom_stopwords)

# Lemmatizer
lemmatizer = WordNetLemmatizer()


def clean_text(text):
    """
    Clean and preprocess a text string.
    """

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove punctuation and numbers
    text = re.sub(r"[^a-z\s]", " ", text)

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenise
    words = text.split()

    # Remove stopwords and very short words
    words = [
        word
        for word in words
        if word not in stop_words
        and len(word) > 2
    ]

    # Lemmatise
    words = [
        lemmatizer.lemmatize(word)
        for word in words
    ]

    # Return cleaned sentence
    return " ".join(words)


# Apply NLP cleaning
df["clean_text"] = df[text_column].apply(clean_text)


# ============================================================
# 9. DISPLAY ORIGINAL VS CLEANED TEXT
# ============================================================

print("\nOriginal vs cleaned text:")

display(
    df[[text_column, "clean_text"]].head(10)
)


# ============================================================
# 10. REMOVE EMPTY CLEANED TEXT
# ============================================================

df = df[df["clean_text"].str.strip() != ""].copy()

df.reset_index(drop=True, inplace=True)

print("\nFinal dataset shape:")
print(df.shape)


# ============================================================
# 11. TF-IDF VECTORIZATION
# ============================================================

"""
TF-IDF converts the cleaned text into numerical values.

ngram_range=(1,2)
means we use:
    - individual words (unigrams)
    - pairs of words (bigrams)
"""

tfidf = TfidfVectorizer(
    max_features=5000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X = tfidf.fit_transform(df["clean_text"])

print("\nTF-IDF completed.")
print("Number of documents:", X.shape[0])
print("Number of TF-IDF features:", X.shape[1])


# ============================================================
# 12. DISPLAY TF-IDF FEATURES
# ============================================================

feature_names = tfidf.get_feature_names_out()

print("\nFirst 50 TF-IDF features:")
print(feature_names[:50])


# ============================================================
# 13. ELBOW METHOD FOR K-MEANS
# ============================================================

"""
The elbow method helps us decide how many clusters (K)
should be used.
"""

inertia = []

k_range = range(2, 11)

for k in k_range:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    model.fit(X)

    inertia.append(model.inertia_)


# Plot elbow curve
plt.figure(figsize=(10, 6))

plt.plot(
    k_range,
    inertia,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for K-Means")

plt.xticks(list(k_range))

plt.grid(True)

plt.show()


# ============================================================
# 14. SILHOUETTE SCORE
# ============================================================

"""
The silhouette score measures how well-separated the clusters are.

Higher values generally indicate better-defined clusters.
"""

silhouette_scores = []

for k in k_range:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(X)

    score = silhouette_score(
        X,
        labels
    )

    silhouette_scores.append(score)


# Display scores
print("\nSilhouette scores:")

for k, score in zip(k_range, silhouette_scores):
    print(
        f"K = {k}: "
        f"Silhouette Score = {score:.4f}"
    )


# Plot silhouette scores
plt.figure(figsize=(10, 6))

plt.plot(
    k_range,
    silhouette_scores,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score for Different K Values")

plt.xticks(list(k_range))

plt.grid(True)

plt.show()


# ============================================================
# 15. SELECT BEST K
# ============================================================

best_k = list(k_range)[
    np.argmax(silhouette_scores)
]

best_score = max(silhouette_scores)

print("\nRecommended number of clusters:", best_k)
print("Best silhouette score:", round(best_score, 4))


# ============================================================
# 16. RUN FINAL K-MEANS MODEL
# ============================================================

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

# Generate cluster labels
df["cluster"] = kmeans.fit_predict(X)

print("\nK-Means clustering completed.")


# ============================================================
# 17. COUNT RECORDS IN EACH CLUSTER
# ============================================================

cluster_counts = (
    df["cluster"]
    .value_counts()
    .sort_index()
)

print("\nNumber of records in each cluster:")
print(cluster_counts)


# ============================================================
# 18. PLOT CLUSTER SIZES
# ============================================================

plt.figure(figsize=(10, 6))

plt.bar(
    cluster_counts.index.astype(str),
    cluster_counts.values
)

plt.xlabel("Cluster")
plt.ylabel("Number of Dialogues")
plt.title("Number of LOTR Dialogues per Cluster")

plt.grid(axis="y")

plt.show()


# ============================================================
# 19. FIND TOP WORDS FOR EACH CLUSTER
# ============================================================

"""
K-Means creates cluster centres.

The largest values in each cluster centre correspond to
the words that are most strongly associated with that cluster.
"""

terms = tfidf.get_feature_names_out()

order_centroids = (
    kmeans.cluster_centers_
    .argsort(axis=1)[:, ::-1]
)


print("\n" + "=" * 70)
print("TOP WORDS IN EACH CLUSTER")
print("=" * 70)


cluster_keywords = {}

for cluster_number in range(best_k):

    top_words = [
        terms[index]
        for index in order_centroids[
            cluster_number,
            :20
        ]
    ]

    cluster_keywords[cluster_number] = top_words

    print(
        f"\nCluster {cluster_number}:"
    )

    print(
        ", ".join(top_words)
    )


# ============================================================
# 20. CREATE A CLUSTER SUMMARY TABLE
# ============================================================

cluster_summary = []

for cluster_number in range(best_k):

    cluster_data = df[
        df["cluster"] == cluster_number
    ]

    cluster_summary.append({
        "Cluster": cluster_number,
        "Number of Dialogues": len(cluster_data),
        "Top Keywords": ", ".join(
            cluster_keywords[cluster_number][:10]
        )
    })


cluster_summary_df = pd.DataFrame(
    cluster_summary
)

print("\nCluster Summary:")
display(cluster_summary_df)


# ============================================================
# 21. SHOW EXAMPLES FROM EACH CLUSTER
# ============================================================

for cluster_number in range(best_k):

    print("\n")
    print("=" * 80)
    print(f"CLUSTER {cluster_number}")
    print("=" * 80)

    cluster_data = df[
        df["cluster"] == cluster_number
    ]

    # Display up to 10 examples
    examples = cluster_data.head(10)

    for _, row in examples.iterrows():

        print(
            "•",
            row[text_column]
        )


# ============================================================
# 22. PCA DIMENSION REDUCTION
# ============================================================

"""
TF-IDF can contain thousands of dimensions.

PCA reduces the data to 2 dimensions so that we can
visualise the clusters.
"""

# Convert sparse matrix to dense matrix
X_dense = X.toarray()

print("\nOriginal number of dimensions:", X_dense.shape[1])


pca = PCA(
    n_components=2,
    random_state=42
)

X_pca = pca.fit_transform(X_dense)

print(
    "Dimensions after PCA:",
    X_pca.shape[1]
)


# ============================================================
# 23. PCA CLUSTER VISUALISATION
# ============================================================

plt.figure(figsize=(12, 8))

scatter = plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=df["cluster"],
    alpha=0.6
)

plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")

plt.title(
    "LOTR Dialogue Clusters using K-Means + NLP"
)

plt.colorbar(
    scatter,
    label="Cluster"
)

plt.grid(True)

plt.show()


# ============================================================
# 24. ADD PCA VALUES TO DATAFRAME
# ============================================================

df["PCA_1"] = X_pca[:, 0]
df["PCA_2"] = X_pca[:, 1]


# ============================================================
# 25. SHOW FINAL DATASET
# ============================================================

print("\nFinal dataset:")
display(df.head(20))


# ============================================================
# 26. FIND THE MOST REPRESENTATIVE DIALOGUES
# ============================================================

"""
For each cluster, find the dialogue closest to the
cluster centre.

This gives an example of a representative dialogue.
"""

distances = kmeans.transform(X)

representative_rows = []

for cluster_number in range(best_k):

    cluster_indices = np.where(
        df["cluster"].values == cluster_number
    )[0]

    cluster_distances = distances[
        cluster_indices,
        cluster_number
    ]

    closest_index = cluster_indices[
        np.argmin(cluster_distances)
    ]

    representative_rows.append({
        "Cluster": cluster_number,
        "Representative Dialogue":
            df.iloc[closest_index][text_column]
    })


representative_df = pd.DataFrame(
    representative_rows
)

print("\nRepresentative dialogue for each cluster:")

display(representative_df)


# ============================================================
# 27. SAVE CLUSTERED DATASET
# ============================================================

output_file = "LOTR_kmeans_clusters.csv"

df.to_csv(
    output_file,
    index=False
)

print(
    f"\nClustered dataset saved as: {output_file}"
)


# ============================================================
# 28. SAVE CLUSTER SUMMARY
# ============================================================

summary_file = "LOTR_cluster_summary.csv"

cluster_summary_df.to_csv(
    summary_file,
    index=False
)

print(
    f"Cluster summary saved as: {summary_file}"
)


# ============================================================
# 29. SAVE REPRESENTATIVE DIALOGUES
# ============================================================

representative_file = (
    "LOTR_representative_dialogues.csv"
)

representative_df.to_csv(
    representative_file,
    index=False
)

print(
    "Representative dialogues saved as:",
    representative_file
)


# ============================================================
# 30. FINAL RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("FINAL RESULTS")
print("=" * 70)

print(
    "\nOriginal dataset rows:",
    len(df)
)

print(
    "Number of clusters:",
    best_k
)

print(
    "Best silhouette score:",
    round(best_score, 4)
)

print(
    "\nCluster sizes:"
)

print(cluster_counts)

print(
    "\nOutput files:"
)

print("1. LOTR_kmeans_clusters.csv")
print("2. LOTR_cluster_summary.csv")
print("3. LOTR_representative_dialogues.csv")

print("\nAnalysis complete.")
```


SyntaxError: invalid syntax (390140405.py, line 1)